# Pipeline: TIL+ spots and TIL zones

Produces the spot table frozen at `config.TIL_ZONES_CSV` (EFS
`results/visium/til_zones/til_zone_annotations.csv`: spot_id, sample, TIL_positive, TIL_zone), which
every TIL analysis reads through `gbmcollaboration.paper.til` (870,077 spots in 228 sections, 45,504 TIL+).

- **TIL+ spot**: summed cell2location absolute abundance of the level-4 types whose
  `celltype_level2_curated` is `T_NK` is ≥ 2.
- **Zones**, per Visium section, kNN with K = 6 on the spot image coordinates: BZ1 = neighbours of
  TIL+ spots that are not TIL+; BZ2 = neighbours of BZ1 spots that are neither TIL+ nor BZ1; every
  other spot TIL−.

**Origin**: Spencer Watson's EFS `Analysis_Repo/Notebooks/TIL_Niche_Comparisons_v2_clean.ipynb`, cells
21 (TIL+ definition; cell 19 is an identical copy), 22 (zones) and 24 (write the table), with their
markdown. Changes:
- Inputs through the package instead of the notebook's loading cells: spots and image coordinates
  from `AllDataCenters.load_spatial(load_gex=False)` (same spots and coordinates as
  `load_spatial_adata`, without the > 100 GB dense expression matrix these cells do not use);
  cell2location abundances from `visium.load_c2l()` + `visium.add_c2l_to_obs()`; the annotation key
  from `annotation.add_celltype_key(adata, efs_copy=True)` (the EFS copy the notebook read). `adata`
  holds only `obs` and the key.
- Cell 24 writes to this notebook's output folder unless `RUN_PIPELINE = True` (then to the frozen
  path), and the result is compared with the frozen file.
- In the source notebook cells 22 and 24 were not executed: the frozen file was written by an
  earlier run. Re-run here, the table is identical to the frozen file (checked at the end).

In [ ]:
import gc
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors

from gbmcollaboration import AllDataCenters
from gbmcollaboration.paper import annotation, config, output, visium

In [ ]:
# Parameters
RUN_PIPELINE = False   # True overwrites the frozen config.TIL_ZONES_CSV
OUT = output.setup("pipelines_TIL_zones")

### Inputs: spots, image coordinates, cell2location abundances, annotation key

In [ ]:
loaded = AllDataCenters.load_spatial(
    exclude_low_quality_visium=True, **config.LOADER_FLAGS, list_sample_id=None,
    normalization="log_norm_count", radius=1, deconvolution_slot_name="deconv_cohort_level_4",
    load_gex=False, load_deconvolution=True,
)
obs = pd.concat([o.celltype_df[["imagecol", "imagerow"]] for o in loaded.values()])
obs["sample"] = obs.index.get_level_values(0)
obs.index = [f"{s}_{b}" for s, b in obs.index]
del loaded
gc.collect()

adata = ad.AnnData(obs=obs)
visium.add_c2l_to_obs(adata, visium.load_c2l(sorted(adata.obs["sample"].unique())))
annotation.add_celltype_key(adata, efs_copy=True)
print(f"{adata.n_obs:,} spots, {adata.obs['sample'].nunique()} sections")

### Set TIL definition and minimum threshold

In [ ]:
LEVEL_COL     = "celltype_level2_curated"
TIL_GROUPS    = ["T_NK"]
TIL_THRESHOLD = 2

key = adata.uns["celltype_annotation_key"]

# level 4 types that belong to T_NK at level2_curated
lvl4_index = key.index.astype(str)
lvl_values = key[LEVEL_COL].astype(str)
til_level4 = lvl4_index[lvl_values.isin(TIL_GROUPS)]

# corresponding C2L columns
til_c2l_cols = [f"meanscell_abundance_w_sf_{ct}" for ct in til_level4]

# per spot TIL_count and TIL_positive
adata.obs["TIL_count"] = adata.obs[til_c2l_cols].sum(axis=1)
adata.obs["TIL_positive"] = adata.obs["TIL_count"] >= TIL_THRESHOLD

# per-sample summary (using 'sample' as the grouping key)
til_stats = adata.obs.groupby("sample")["TIL_positive"].agg(
    TIL_spots="sum",
    total_spots="count",
)
til_stats["TIL_fraction"] = til_stats["TIL_spots"] / til_stats["total_spots"]

### Define TIL niche zones

In [ ]:
# Define TIL zones (TIL+, BZ1, BZ2, TIL-) per sample using kNN in spatial coordinates

# parameters
K = 6
KEY = "patient" if "patient" in adata.obs.columns else "sample"
COORD_COLS = ["imagecol", "imagerow"] if {"imagecol", "imagerow"}.issubset(adata.obs.columns) else None

# guard: require TIL_positive
if "TIL_positive" not in adata.obs.columns:
    raise KeyError("TIL_positive not found in adata.obs. Compute TIL_positive first.")

# initialize zones from TIL_positive
zones = pd.Series("TIL-", index=adata.obs_names)
zones.loc[adata.obs["TIL_positive"].astype(bool)] = "TIL+"

# coordinate getter
def get_coords(index):
    if COORD_COLS is not None:
        xy = adata.obs.loc[index, COORD_COLS].to_numpy(dtype=float)
    elif "spatial" in adata.obsm_keys():
        pos = adata.obs.index.get_indexer(index)
        xy = np.asarray(adata.obsm["spatial"][pos], dtype=float)
    else:
        raise KeyError("No coordinates found (need obs['imagecol','imagerow'] or obsm['spatial']).")
    return xy

# per-sample kNN to define BZ1 and BZ2
for s, idx in adata.obs.groupby(KEY).groups.items():
    idx = pd.Index(idx)
    tilp = (zones.loc[idx] == "TIL+").to_numpy()
    if not tilp.any():
        continue

    xy = get_coords(idx)

    # drop rows with non-finite coordinates
    mask = np.isfinite(xy).all(1)
    if mask.sum() < 2:
        continue
    idx_m  = idx[mask]
    xy_m   = xy[mask]
    tilp_m = tilp[mask]

    # kNN (k includes self; we drop self in neighbor lists)
    k = int(min(K + 1, len(idx_m)))
    nn = NearestNeighbors(n_neighbors=k, metric="euclidean").fit(xy_m)
    _, nbrs = nn.kneighbors(xy_m)

    # first border zone (BZ1): neighbors of TIL+ that are not themselves TIL+
    bz1 = np.zeros(len(idx_m), dtype=bool)
    for i in np.where(tilp_m)[0]:
        n = nbrs[i]
        n = n[n != i]
        bz1[n] = True
    bz1 &= ~tilp_m

    # second border zone (BZ2): neighbors of BZ1 that are not TIL+ or BZ1
    bz2 = np.zeros(len(idx_m), dtype=bool)
    for i in np.where(bz1)[0]:
        n = nbrs[i]
        n = n[n != i]
        bz2[n] = True
    bz2 &= ~(tilp_m | bz1)

    zones.loc[idx_m[bz1]] = "BZ1"
    zones.loc[idx_m[bz2]] = "BZ2"

# set ordered categorical for consistent downstream use
adata.obs["TIL_zone"] = pd.Categorical(zones, categories=["TIL+", "BZ1", "BZ2", "TIL-"])
print(adata.obs["TIL_zone"].value_counts().to_string())

### Save TIL spot annotations

In [ ]:
# -------------------------------------------------------------------------
# Paths (adjust if needed)
# -------------------------------------------------------------------------
# frozen file only with RUN_PIPELINE = True; otherwise this notebook's output folder
OUT_CSV = config.TIL_ZONES_CSV if RUN_PIPELINE else OUT / "til_zone_annotations.csv"

# -------------------------------------------------------------------------
# Build spot-level TIL zone table
# -------------------------------------------------------------------------
if "TIL_zone" not in adata.obs.columns:
    raise KeyError("adata.obs['TIL_zone'] not found. Run the TIL zone definition cell first.")

df_til = adata.obs.copy()

# start with index as spot id
df_til = df_til.reset_index().rename(columns={"index": "spot_id"})

# pick useful columns if present
cols = ["spot_id"]
for col in ["sample", "patient", "TIL_positive", "TIL_zone"]:
    if col in df_til.columns:
        cols.append(col)

df_til = df_til[cols]

# -------------------------------------------------------------------------
# Save
# -------------------------------------------------------------------------
df_til.to_csv(OUT_CSV, index=False)

print("Wrote TIL zone annotations to:")
print(" ", OUT_CSV)
print("\nPreview:")
print(df_til.head())

In [ ]:
# Compared with the frozen file
frozen = pd.read_csv(config.TIL_ZONES_CSV).set_index("spot_id")
new = df_til.set_index("spot_id")
print(f"spots: frozen {len(frozen):,}, this run {len(new):,}, same set {set(frozen.index) == set(new.index)}")
common = new.index.intersection(frozen.index)
for col in ["sample", "TIL_positive", "TIL_zone"]:
    n = int((new.loc[common, col].astype(str).str.lower() != frozen.loc[common, col].astype(str).str.lower()).sum())
    print(f"{col}: {n} spots differ")
print(f"TIL+ spots: {int(new['TIL_positive'].sum()):,}")

In [ ]:
print(output.versions(["numpy", "pandas", "scikit-learn", "anndata", "s3fs"]).to_string(index=False))